# Prompt validation — the two-sided test

A detector is only meaningful if it does **both**: finds what is there, and stays
quiet on what is not. Recall alone rewards a model that reports something for every
input; precision alone rewards one that reports nothing.

This runs the toy fixture (2 planted cross-file vulnerabilities) and a clean repo
(no such vulnerability), then reports a three-line verdict.

**T4 GPU → Runtime → Restart session before running.**

In [ ]:
#@title 1. Install — version check
import sys
assert "wca" not in sys.modules, "Stale module. Runtime > Restart session."

REPO = "https://github.com/quinyang/whole_codebase_auditor"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

!pip install -q --upgrade --force-reinstall --no-deps "wca @ git+{REPO}@{BRANCH}"
!pip install -q "wca[gpu] @ git+{REPO}@{BRANCH}"

import wca
print("wca", wca.__version__)
MIN = (0, 5, 1)
assert tuple(map(int, wca.__version__.split("."))) >= MIN, (
    f"got {wca.__version__}, need >= 0.5.1 -- the push did not land"
)

In [ ]:
#@title 2. Run both controls (~6 min)
from wca.infer import load_auditor
from wca.sweep import run_validation

auditor = load_auditor()   # cached: re-running this cell reuses the model          # ~6 GiB allocated means 4-bit worked
result = run_validation(auditor, negative_control="pallets/click", budget=4000)

In [ ]:
#@title 3. Save
import json, os
os.makedirs("/content/wca_runs", exist_ok=True)
with open("/content/wca_runs/validation.json", "w") as fh:
    json.dump(result, fh, indent=2)
print("saved -> /content/wca_runs/validation.json")

## Reading the verdict

| Result | Meaning | Next |
|---|---|---|
| all three PASS | prompt is sound | session 3: build the benchmark |
| recall FAIL | missed planted vulns in a 640-token context | prompt or model capability, not context |
| grounding FAIL | still authoring evidence | replace `evidence` text with a line NUMBER |
| precision FAIL | false positives on a clean repo | fix before scaling — this sinks a precision claim |

A **0 findings** result on `pallets/click` is a PASS, not a failure. Click is an
argument parser — no database, no credentials, no network sink. The single finding
your earlier run produced was a false positive that prompt v1 pressured out of the model.